# Test Epic Clinical Notes Appointments Get Notebook

This notebook tests the epic clinical notes appointments data extraction pipeline for pat2vec, specifically testing:

## What's Tested

1. **Elasticsearch Integration**
   - Elasticsearch container startup and health check
   - Index refresh for `epic_epic_clinical_notes_appointments` (clinical notes appointments data source)
   - Elasticsearch credentials setup

2. **Database Backend Processing**
   - SQLite database creation and connection
   - Raw clinical notes appointments data fetched from Elasticsearch
   - Data saved to `raw_data_raw_epic_clinical_notes_appointments` table

3. **pat_maker Feature Extraction**
   - Patient batch processing with clinical notes appointments enabled (`main_options={"epic_epic_clinical_notes_appointments": True}`)
   - Time window slicing and feature vector generation
   - Database storage of processed features in `features_features` table

4. **Merge Function Verification**
   - `merge_epic_clinical_notes_appointments_csv()` processes all patients
   - Merged CSV output to `merged_batches/merged_epic_clinical_notes_appointments.csv`
   - Data integrity checks on merged output

## Expected Outputs

| Output | Location | Expected Rows |
|--------|----------|---------------|
| Raw clinical notes appointments data | `raw_data_raw_epic_clinical_notes_appointments` table | ≥0 (depends on dummy data) |
| Merged CSV | `merged_batches/merged_epic_clinical_notes_appointments.csv` | Same as raw, with patient IDs merged |
| Features output | `features_features` table | 1 row per time slice per patient |

## Database Table Issues (Expected)

The following tables may NOT be populated because they are not enabled in this test:
- `raw_data_raw_news`
- `raw_data_raw_bloods`
- `raw_data_raw_diagnostics`
- `raw_data_raw_drugs`
- `raw_data_raw_demographics`

These errors in the pat_maker logs during batch fetching are **expected** and do not indicate failure
since only epic clinical notes appointments (`"epic_epic_clinical_notes_appointments": True`) is enabled in this test. The notebook will still pass
if epic_clinical_notes_appointments-specific data is correctly fetched and merged.

## Success Criteria

- Clinical notes appointments rows retrieved from `raw_data_raw_epic_clinical_notes_appointments` table ✓
- Expected columns present in raw clinical notes appointments data ✓
- Merged CSV file created with non-zero row count ✓
- All temporary files cleaned up (no residual project dir/database/credentials) ✓

In [ ]:
import os
import random
import shutil
import sys

import numpy as np

In [ ]:
random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))
print(f"Pat2vec path: {pat2vec_dir}")

In [ ]:
dir_to_remove = "epic_clinical_notes_appointments_test_project"
try:
    shutil.rmtree(dir_to_remove, ignore_errors=True)
except Exception as e:
    msg = f"Failed to clean up directory: {e}."
    raise RuntimeError(msg)

print("Previous outputs cleaned.")

In [ ]:
from pat2vec.util.docker_elastic import ElasticContainer

es_container = ElasticContainer()
es_container.stop()

print("Starting Elasticsearch container...")
if not es_container.start():
    msg = "Failed to start Elasticsearch."
    raise RuntimeError(msg)

host, username, password = es_container.get_credentials()

creds_filename = "test_elastic_credentials_epic_clinical_notes_appointments_get.py"
creds_content = f"""\nusername = '{username}'\npassword = '{password}'\napi_key = None\nhosts = ["{host}"]\n"""

with open(creds_filename, "w") as f:
    f.write(creds_content)

print(f"Created {creds_filename}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

grandparent_dir = "/workspaces/pat2vec"
schema_path = os.path.join(grandparent_dir, "test_files", "elastic_schemas.json")

config_populate = config_class(
    proj_name="epic_clinical_notes_appointments_test_project",
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
    all_patient_list=["PXAJI0Y6", "PDPBHSAH", "PXTHV3A3", "PZMF8MDD", "P4V30T9N"],
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print(f"Population complete. Generated {len(patient_ids)} dummy patients.")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = [
    "epr_documents",
    "basic_observations",
    "observations",
    "order",
    "pims_apps",
    "epic_clinical_notes_appointments",
]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
import time

time.sleep(2)
print("Indices refreshed.")

In [ ]:
PROJ_NAME = "epic_clinical_notes_appointments_test_project"
DB_FILENAME = "temp_epic_encounters_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    msg = f"Failed to remove database: {e}."
    raise RuntimeError(msg)

db_connection_string = "sqlite:///" + DB_PATH
print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.main_pat2vec import main

try:
    config_obj = config_class(
        proj_name=PROJ_NAME,
        credentials_path=creds_filename,
        current_path_dir="",
        main_options={"epic_clinical_notes_appointments": True},
        batch_mode=True,
        verbosity=0,
        random_seed_val=random_seed_value,
        testing=True,
        testing_elastic=True,
        dummy_medcat_model=True,
        use_controls=False,
        medcat=False,
        start_time=None,
        patient_id_column_name="client_idcode",
        annot_filter_options={},
        shuffle_pat_list=False,
        storage_backend="database",
        db_connection_string=db_connection_string,
        check_patient_existence=False,
        all_patient_list=["PXAJI0Y6", "PDPBHSAH", "PXTHV3A3", "PZMF8MDD", "P4V30T9N"],
    )
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except Exception as e:
    msg = f"Failed to initialize: {e}"
    raise RuntimeError(msg)

In [ ]:
print("\n=== PROCESSING PATIENTS WITH pat_maker ===")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

try:
    print(f"Processing patient 0: {pat2vec_obj.all_patient_list[0]}")
    pat2vec_obj.pat_maker(0)
except Exception as e:
    msg = (
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features."
    )
    raise RuntimeError(
        msg,
    ) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    msg = (
        "FATAL ERROR: get_all_features returned an empty DataFrame. "
        "This indicates a critical failure in the pat2vec pipeline. "
        "No features were extracted or saved to the database."
    )
    raise RuntimeError(
        msg,
    )

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
import os

import pandas as pd

from pat2vec.util.post_processing_build_methods import (
    merge_epic_clinical_notes_appointments_csv,
)


def merge_epic_clinical_notes_appointments_data(all_pat_list, config_obj):
    """Merge all Epic clinical notes appointments data for the patient list.

    Raises:
        ValueError: If no data is returned after merging.

    """
    merged_path = merge_epic_clinical_notes_appointments_csv(
        all_pat_list,
        config_obj,
        overwrite=True,
    )

    if not os.path.exists(merged_path):
        msg = f"MERGE FAILED: Merged file not found at {merged_path}. No epic_clinical_notes_appointments data was returned."
        raise ValueError(
            msg,
        )

    merged_df = pd.read_csv(merged_path)

    if len(merged_df) == 0:
        msg = "MERGE FAILED: Merged epic_clinical_notes_appointments file is empty. No epic_clinical_notes_appointments data was returned."
        raise ValueError(
            msg,
        )

    print("Merged epic_clinical_notes_appointments saved to: " + merged_path)
    print(f"Merge successful: {len(merged_df)} rows")

    return merged_path


all_pat_list = pat2vec_obj.all_patient_list
print("\n=== BUILDING MERGED EPIC CLINICAL NOTES APPOINTMENTS ===")
try:
    merged_path = merge_epic_clinical_notes_appointments_data(all_pat_list, config_obj)
except Exception as e:
    msg = f"Failed to build merged epic_clinical_notes_appointments: {e}"
    raise RuntimeError(
        msg,
    ) from e

In [ ]:
print("\n=== DATABASE AND PROJECT CLEANUP ===")

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Removed database: {DB_PATH}")
except Exception as e:
    msg = f"Failed to remove database file '{DB_PATH}': {e}. Critical error - cleanup incomplete."
    raise RuntimeError(
        msg,
    ) from e

try:
    if os.path.exists(PROJ_NAME):
        shutil.rmtree(PROJ_NAME, ignore_errors=False)
        print(f"Removed project directory: {PROJ_NAME}")
except Exception as e:
    msg = f"Failed to remove '{PROJ_NAME}' directory: {e}. Critical error - cleanup incomplete."
    raise RuntimeError(
        msg,
    ) from e

try:
    if os.path.exists(creds_filename):
        os.remove(creds_filename)
        print(f"Removed Elasticsearch credentials: {creds_filename}")
except Exception as e:
    msg = (
        f"Failed to remove Elasticsearch credentials file '{creds_filename}': {e}. "
        "Critical error - cleanup incomplete."
    )
    raise RuntimeError(
        msg,
    ) from e

In [ ]:
print("\n=== FINAL VERIFICATION ===")

assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(PROJ_NAME), "Project directory still exists!"
assert not os.path.exists(
    creds_filename,
), "Elasticsearch credentials file still exists!"

print("All cleanup verified - no residual files remain.")
print("\n=== TEST SUCCESSFUL ===")